Enter OpenAI API Key Path

In [ ]:
api_key_path = "/Users/cu135/Library/CloudStorage/OneDrive-Personal/OneDrive_Documents/Work/Software/OpenAI/cu135_cbct_key.txt"

01 - Generate JSON from Post-Inclusion/Exclusion CSV
- Enter path to CSV generated from Notebook 04 (inclusion/exclusion CSV)
- Enter path to JSON which has the articles with labeled sections

In [ ]:
csv_path = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/resources/datasets/TMS_studies_influencing_memory/metadata/Updated_TMS_studies_influencing_memory.csv'
json_file_path = "/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/resources/datasets/TMS_studies_influencing_memory/metadata/json/_other_labeled_sections.json"

In [ ]:
from calvin_utils.gpt_sys_review.json_utils import FilterPapers

# Initialize and run the FilterPapers class
filter_papers = FilterPapers(csv_path=csv_path, json_path=json_file_path)
filtered_json_path = filter_papers.run()

02 - Prepare Data Extraction Questions
- Dev Note: This is a repeat class and should be inherited from a .py

In [ ]:
from calvin_utils.gpt_sys_review.examples.question_utils import QuestionTemplate

question_template = QuestionTemplate()
question_template.data_extraction_questions()

Copy the Template Dict from Above and Fill it Out As per the Example

In [ ]:
question = {
    "What was the duration between TMS and measurement? How many days, weeks, or months apart were the initial and post-TMS evaluations?": "intertest_interval",
    "When was the TMS administered relative to mmory? Was it during encoding (presentation of stimulus), in consolidation (after stimulus), during recall (during recall), or priming (such as before testing, like when TMS is given every day for weeks between sessions).": "memory_timing",
    }

03 - Extract Data with GPT

Define the segmented labels you want to consider. 

- Article type 'case' will has sections 'case_report' and 'other'
- Article type 'research' has sections "Abstract", "Introduction", "Methods", "Results", "Discussion", "Conclusion", "References"


In [ ]:
# Define the keys you want to consider (exclude 'References')
keys_to_consider = ["Positive"]  # Add or remove keys as per your requirement

In [ ]:
test_mode=False

In [ ]:
from calvin_utils.gpt_sys_review.gpt_utils.openai_json_evaluator import OpenAIJsonEvaluator
evaluator = OpenAIJsonEvaluator(api_key_path=api_key_path,
                                json_file_path=json_file_path, 
                                keys_to_consider=keys_to_consider,
                                question_type="extraction",
                                question=question,
                                test_mode=test_mode,
                                model_choice="gpt4",
                                debug=False)
answers = evaluator.evaluate_all_files()
evaluated_json_path = evaluator.save_to_json(answers)

04 - Convert results to a CSV
- Set answers_binary to False if the questions you asked do not have binary answers. 
   - We will extract the raw data, like specific result values, for you to review.
- Set asnwers_binary to True if the questions you asked do have binary answers. 
   - By default, we will set positive answers to 1, and negative answers to 0.

In [ ]:
answers_binary=False

In [ ]:
api_key_path = "/Users/cu135/Library/CloudStorage/OneDrive-Personal/OneDrive_Documents/Work/Software/OpenAI/cu135_cbct_key.txt"

In [ ]:
from calvin_utils.gpt_sys_review.json_utils import CustomSummarizer
custom_summarizer = CustomSummarizer(json_path=evaluated_json_path, answers_binary=answers_binary, summary_type='llm', api_key_path=api_key_path)
df, raw_path, automated_path = custom_summarizer.run_custom()

05 - Postprocess

- If you have been using a master_list, you can update it with the results from the generated CSVs.

In [ ]:
master_list_path = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/resources/datasets/TMS_studies_influencing_memory/metadata/Updated_TMS_studies_influencing_memory_final.csv'

In [ ]:
from calvin_utils.gpt_sys_review.txt_utils import PostProcessing
PostProcessing.add_raw_results_to_master_list(master_list_path=master_list_path, raw_results_path=raw_path)

Your articles have been completely evaluated. 

Please check the CSVs in the directory noted above and use the path to the one you would like to use for your further analysis.
- Enjoy. If this has been helpful, please consider adding Calvin Howard as a collaborator. 
- e: choward12@bwh.harvard.edu